<div align="center" dir="rtl" style="direction:rtl;text-align:center;">

<p><img src="../src/logo.png" alt="لوگوی مجموعهٔ آزمون‌های آزمایشی هوش مصنوعی" width="400"></p>
<p><strong>مجموعهٔ آزمون‌های آزمایشی هوش مصنوعی</strong></p>
<p>آزمون مرحله 3 دوره دوم</p>
<h1>ردپای سایه‌ها</h1>
<p>سطح ۲ · طبقه‌بندی، KNN و PCA از صفر</p>
<p><strong>کوسه‌ها پیش از نخستین درختان روی زمین زندگی می‌کردند.</strong></p>

</div>

<div dir="rtl" align="right" style="direction:rtl;text-align:right;">

## مأموریت شما

در بانک سپهر، سامانهٔ «ردپای سایه‌ها» باید تراکنش‌های مشکوک را پیش از تأیید نهایی پیدا کند. هر تراکنش با نشانه‌های رفتاری و فنی توصیف شده است؛ اما این نشانه‌ها مقیاس‌های متفاوتی دارند و همه به یک اندازه برای تشخیص تقلب مفید نیستند.

مدلی بسازید که برچسب `is_fraud` را پیش‌بینی کند: `0` یعنی تراکنش سالم و `1` یعنی تقلب. در این تمرین، KNN و PCA را خودتان با NumPy پیاده‌سازی می‌کنید، اثر مقیاس ویژگی‌ها را می‌سنجید و با بررسی داده تصمیم می‌گیرید کدام ویژگی‌ها به مدل کمک می‌کنند.

### داده‌ها

| فایل | محتوا |
|---|---|
| `data/train/train.csv` | ۱۶۸۰ تراکنش برچسب‌دار |
| `data/validation/validation.csv` | ۳۶۰ تراکنش برچسب‌دار |
| `data/test/test.csv` | ۳۶۰ تراکنش بدون برچسب |

ورودی‌ها ۱۸ ستون‌اند: ۱۲ امتیاز رفتاری/تراکنشی (`transaction_amount`, `login_failed_score`, `geo_velocity_score`, `unusual_merchant_score`, `new_payee_score`, `off_hours_activity_score`, `card_testing_score`, `cross_border_score`, `beneficiary_risk_score`, `session_risk_score`, `device_risk_score`, `account_activity_score`) و ۶ ویژگی فنی نشست (`session_duration`, `page_load_time_ms`, `payload_size_bytes`, `network_jitter_ms`, `cache_miss_count`, `api_response_time_ms`). ستون `id` فقط شناسه است و نباید وارد مدل شود. کلاس‌ها نامتوازن‌اند؛ بنابراین Accuracy معیار مناسبی برای مقایسهٔ تنها نیست.

### کارهای مورد انتظار

1. تفاوت مقیاس و واریانس ویژگی‌ها و همبستگی آن‌ها با هدف را بررسی کنید.
2. `ScratchKNNClassifier` را با فاصلهٔ اقلیدسی و رأی همسایه‌ها از صفر بسازید.
3. `ScratchPCA` را با تجزیهٔ مقادیر ویژهٔ ماتریس کوواریانس پیاده‌سازی کنید.
4. یک baseline بسازید، سپس اثر نرمال‌سازی، تعداد مؤلفه‌های PCA و انتخاب ویژگی را روی validation مقایسه کنید.
5. در صورت نیاز آستانهٔ تصمیم را با Macro-F1 تنظیم کنید؛ فقط نتیجهٔ validation را برای انتخاب روش به‌کار ببرید.

### محدودیت‌ها

کتابخانه‌های مجاز: Python استاندارد، `numpy`, `pandas`, `matplotlib`. استفاده از `sklearn`، پیاده‌سازی آمادهٔ KNN/PCA، اینترنت یا فایل‌های خارج از پوشهٔ تسک مجاز نیست. KNN و PCA باید با NumPy پیاده‌سازی شوند.

### معیار و نمره‌دهی

معیار **Macro-F1** میانگین F1 کلاس سالم و کلاس تقلب است. نمره از ۰ تا ۲۰ محاسبه می‌شود: کمتر از baseline برابر `0.45` نمره صفر؛ رسیدن به هدف `0.90` نمره ۲۰؛ بین این دو `round(5 + 15 * (Macro-F1 - 0.45) / (0.90 - 0.45))` با سقف ۱۹.

### قالب ارسال

`submission.csv` باید دقیقاً دو ستون `id,prediction` داشته باشد. برای هر `id` تست یک پیش‌بینی دودویی (`0` یا `1`) ثبت کنید. آرایهٔ `predictions` در سلول پایانی به این قالب تبدیل می‌شود.

</div>

<div dir="rtl" align="right" style="direction:rtl;text-align:right;">

## ۱. بارگذاری و بررسی داده‌ها

داده‌ها را بخوانید و تعداد هر کلاس، دامنه و انحراف معیار ستون‌ها را بررسی کنید. شناسه و برچسب را از ماتریس ویژگی جدا نگه دارید.

</div>

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA_DIR=Path("data")
train_df=pd.read_csv(DATA_DIR/"train"/"train.csv")
validation_df=pd.read_csv(DATA_DIR/"validation"/"validation.csv")
test_df=pd.read_csv(DATA_DIR/"test"/"test.csv")
FEATURE_COLS=[c for c in train_df.columns if c not in ("id","is_fraud")]
BEHAVIORAL_COLS=["transaction_amount","login_failed_score","geo_velocity_score","unusual_merchant_score","new_payee_score","off_hours_activity_score","card_testing_score","cross_border_score","beneficiary_risk_score","session_risk_score","device_risk_score","account_activity_score"]
TECHNICAL_COLS=[c for c in FEATURE_COLS if c not in BEHAVIORAL_COLS]
y_train=train_df["is_fraud"].to_numpy(int)
print("shapes:",train_df.shape,validation_df.shape,test_df.shape)
print("class counts:",np.bincount(y_train))
print(train_df[FEATURE_COLS].describe().T[["mean","std","min","max"]].round(2))


<div dir="rtl" align="right" style="direction:rtl;text-align:right;">

## ۲. مقیاس‌بندی ویژگی‌ها

فاصلهٔ اقلیدسی به واحدهای اندازه‌گیری حساس است. مقیاس‌گذار را با NumPy پیاده‌سازی کنید و میانگین و انحراف معیار را فقط از train بیاموزید.

</div>

In [ ]:
class ScratchStandardScaler:
    def fit(self,X):
        # TODO: Store column means and standard deviations; replace zero scales with one.
        raise NotImplementedError
    def transform(self,X):
        # TODO: Apply stored statistics without refitting on query data.
        raise NotImplementedError


<div dir="rtl" align="right" style="direction:rtl;text-align:right;">

## ۳. پیاده‌سازی KNN از صفر

برای هر نمونهٔ query فاصلهٔ اقلیدسی تا نمونه‌های train را محاسبه کنید و برچسب همسایه‌های نزدیک را به رأی تبدیل کنید. آستانهٔ رأی را طوری طراحی کنید که بعداً قابل تنظیم باشد.

</div>

In [ ]:
class ScratchKNNClassifier:
    def __init__(self,k=5,threshold=0.5):
        self.k=k; self.threshold=threshold; self.X_train=None; self.y_train=None
    def fit(self,X,y):
        # TODO: Store numeric arrays for later distance calculations.
        raise NotImplementedError
    def predict_proba(self,X):
        # TODO: Return the fraction of positive labels among the k nearest neighbors.
        raise NotImplementedError
    def predict(self,X):
        # TODO: Convert positive-class fractions to binary labels using threshold.
        raise NotImplementedError


<div dir="rtl" align="right" style="direction:rtl;text-align:right;">

## ۴. پیاده‌سازی PCA از صفر

میانگین را کم کنید، ماتریس کوواریانس را بسازید، مقادیر و بردارهای ویژه را بیابید و مؤلفه‌ها را بر اساس مقدار ویژه مرتب کنید.

</div>

In [ ]:
class ScratchPCA:
    def __init__(self,n_components):
        self.n_components=n_components; self.mean_=None; self.components_=None; self.explained_variance_ratio_=None
    def fit(self,X):
        # TODO: Center X, eigendecompose its covariance, and retain top components.
        raise NotImplementedError
    def transform(self,X):
        # TODO: Project centered data onto the stored principal components.
        raise NotImplementedError
    def fit_transform(self,X):
        self.fit(X); return self.transform(X)


<div dir="rtl" align="right" style="direction:rtl;text-align:right;">

## ۵. معیارهای ارزیابی

Macro-F1 را با محاسبهٔ F1 هر کلاس بسنجید. هنگام مقایسهٔ مدل‌ها، نتیجهٔ هر دو کلاس را در نظر بگیرید.

</div>

In [ ]:
def macro_f1(y_true,y_pred):
    y_true=np.asarray(y_true,int); y_pred=np.asarray(y_pred,int); scores=[]
    for cls in (0,1):
        tp=np.sum((y_true==cls)&(y_pred==cls)); fp=np.sum((y_true!=cls)&(y_pred==cls)); fn=np.sum((y_true==cls)&(y_pred!=cls))
        scores.append(0.0 if 2*tp+fp+fn==0 else 2*tp/(2*tp+fp+fn))
    return float(np.mean(scores))

def plot_confusion(y_true,y_pred):
    cm=np.zeros((2,2),dtype=int)
    for a,b in zip(y_true,y_pred): cm[int(a),int(b)]+=1
    fig,ax=plt.subplots(figsize=(4,4)); im=ax.imshow(cm,cmap="Blues")
    ax.set(xticks=[0,1],yticks=[0,1],xlabel="Predicted",ylabel="Actual",title="Confusion matrix")
    for i in range(2):
        for j in range(2): ax.text(j,i,str(cm[i,j]),ha="center",va="center")
    fig.colorbar(im,ax=ax); plt.show()


<div dir="rtl" align="right" style="direction:rtl;text-align:right;">

## ۶. baseline: KNN روی همهٔ ویژگی‌ها

یک baseline با مقیاس‌بندی همهٔ ویژگی‌ها و KNN بسازید. سپس آن را با validation ارزیابی کنید.

</div>

In [ ]:
# TODO: Build a validation baseline using all features, scaling, and KNN.


<div dir="rtl" align="right" style="direction:rtl;text-align:right;">

## ۷. کاهش بعد و ریشه‌یابی

PCA را روی همهٔ ویژگی‌ها امتحان کنید. واریانس، loadings و نتیجهٔ validation را بررسی کنید؛ آیا هر مؤلفهٔ پر واریانس هم برای تشخیص تقلب مفید است؟

</div>

In [ ]:
def choose_features():
    # TODO: Return a justified feature list based on your analysis.
    raise NotImplementedError

def fit_pipeline(train_frame,feature_cols,k,n_components,threshold):
    # TODO: Fit scaler, PCA, and KNN on training data; return all fitted objects.
    raise NotImplementedError

def pipeline_predict(state,frame):
    # TODO: Apply the fitted transforms and classifier to new rows.
    raise NotImplementedError


<div dir="rtl" align="right" style="direction:rtl;text-align:right;">

## ۸. بهبود خط‌لوله

بر اساس آمار و نمودارها، گروه ویژگی‌های مفید را انتخاب کنید. تعداد مؤلفه‌ها، `k` و آستانهٔ رأی را فقط با validation انتخاب کنید.

</div>

In [ ]:
# TODO: Tune k, PCA dimension, and threshold using validation Macro-F1 only.


<div dir="rtl" align="right" style="direction:rtl;text-align:right;">

## ۹. اعتبارسنجی نهایی

مدل انتخاب‌شده را روی train آموزش دهید و Macro-F1 اعتبارسنجی را گزارش کنید. برای ثبت نمرهٔ کامل، هدف این تمرین Macro-F1 برابر ۰٫۹۰ است.

</div>

In [ ]:
# TODO: Refit the selected pipeline using train + validation, then predict test.
# Store one binary prediction per test row in `predictions`.
raise NotImplementedError
